In [0]:
-- ============================================================
-- GOLD DIMENSION: dim_channel
-- PURPOSE:
--   Store unique marketing channel combinations based on UTM
--   parameters.
--
-- PRIMARY KEY:
--   channel_id
--
-- RELATIONSHIP:
--   fact_marketing_performance.channel_id
--     -> dim_channel.channel_id
--
-- SOURCE:
--   shoply_analytics.silver.sessions
--
-- NOTE:
--   Null or blank UTM values are mapped to 'unknown' so the
--   dimension has a readable "unknown attribution" member
--   instead of a raw null row.
-- ============================================================

USE CATALOG shoply_analytics;
USE SCHEMA gold;

CREATE OR REPLACE VIEW dim_channel AS
WITH distinct_channels AS (
    SELECT DISTINCT
        COALESCE(NULLIF(utm_source, ''), 'unknown') AS utm_source,
        COALESCE(NULLIF(utm_medium, ''), 'unknown') AS utm_medium,
        COALESCE(NULLIF(utm_campaign, ''), 'unknown') AS utm_campaign
    FROM shoply_analytics.silver.sessions
)

SELECT
    -- Surrogate primary key
    ROW_NUMBER() OVER (
        ORDER BY utm_source, utm_medium, utm_campaign
    ) AS channel_id,

    -- Natural key attributes
    utm_source,
    utm_medium,
    utm_campaign

FROM distinct_channels;